## Tuning of SiPM cut threshold

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np
import awkward as ak
import os
import pint
import glob
import re
from functools import partial

# I use the SiPM QC Analysis kernel (from .venv/bin)

u = pint.get_application_registry()

from lgdo import lh5
from lgdo.lh5 import LH5Store, LH5Iterator, read, read_as, ls, show
from dspeed.vis.waveform_browser import WaveformBrowser
from legendmeta import LegendMetadata

from latools.utils import get_key_for_rawid, get_detector_system_for_channelname, get_filtered_keys_in_detectorsystem
from latools import core
from latools.browse import BrowseTask, BrowseAnydetTask
from latools.histogram import HistogramTask, Histogram2DTask, CategoricalHistogramTask, CategoricalHistogram2DTask, MultiarrayCategoricalHistogramTask
from latools.counter import CountTask

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["font.size"] = 14

proj_dir = "/mnt/atlas02/projects/legend/sipm_qc"
lmeta = LegendMetadata(os.path.join(proj_dir, "metadata/legend-metadata-schwarz"))
#chmap = lmeta.channelmap("20250901T023238Z")

In [ ]:
def get_timestamp_from_filename(filename: str) -> str | None:
    match = re.search(r"\d{8}T\d{6}Z", filename)
    return match.group(0) if match else None

In [ ]:
# custom_chains/dsp-002.yaml
#dsp_dir = os.path.join(proj_dir, "manual_dsp/generated/p14r006dsp-dsp002")
dsp_dir = os.path.join(proj_dir, "manual_dsp/generated/tier/dsp/phy/p16/r000")
dsp_files = glob.glob(dsp_dir+"/l200-*-tier_dsp.lh5")
dsp_files.sort()

#raw_dir = os.path.join(proj_dir, "data/tier/raw/lac/p14/r006")
raw_dir = os.path.join(proj_dir, "data/tier/raw/phy/p16/r000")

chmap = lmeta.channelmap(get_timestamp_from_filename(dsp_files[0]))

def gimme_raw_filename_from_dsp(dspfilename: str):
    return dspfilename.replace(dsp_dir, raw_dir).replace("tier_dsp", "tier_raw")

In [ ]:
def get_name_rawkeys(chmap, *, sort_by_name = True):
    chmap_sipm = chmap.map("system", unique=False).spms
    mapped = chmap_sipm.map(
            "analysis.usability", unique=False).on.map("name")
    ret = [(k, mapped[k].daq.rawid) for k in mapped.keys()]
    if sort_by_name:
        ret = sorted(ret, key=lambda x: x[0])
    return ret


name_rawkeys = get_name_rawkeys(chmap)

# my self-processed dsp files do not have an alias table.
usable_sipm_keys = [f"ch{x[1]}" for x in name_rawkeys]
usable_sipm_names = [x[0] for x in name_rawkeys]
get_name = partial(get_key_for_rawid, chmap)

In [ ]:
dsp_data = {}
for sipm_n, sipm_r in name_rawkeys:
    dsp_data[f"{sipm_n}_wf_lower_hwhm"] = read_as(f"ch{sipm_r}/dsp/wf_lower_hwhm", dsp_files, "ak")
    dsp_data[f"{sipm_n}_wf_mode"] = read_as(f"ch{sipm_r}/dsp/wf_mode", dsp_files, "ak")
    dsp_data[f"{sipm_n}_wf_min_small"] = read_as(f"ch{sipm_r}/dsp/wf_min_small", dsp_files, "ak")
for sipm in usable_sipm_names:
    dsp_data[f"{sipm}_wf_min_rel"] = dsp_data[f"{sipm}_wf_min_small"] - dsp_data[f"{sipm}_wf_mode"]

In [ ]:
def xtalk_stat(data, sipm_names, threshold: float):
    cut_masks = {}
    cut_ratio = {}
    for sipm in sipm_names:
        cut_masks[sipm] = data[f"{sipm}_wf_min_rel"] < -threshold*data[f"{sipm}_wf_lower_hwhm"]
        cut_ratio[sipm] = np.sum(cut_masks[sipm].to_numpy()) / len(cut_masks[sipm])

    cut_aggregate = np.zeros_like(cut_masks[sipm_names[0]], dtype=int).to_numpy()
    for sipm in sipm_names:
        cut_aggregate += cut_masks[sipm].to_numpy().astype(int)

    cut_mult_vals, cut_mult_counts = np.unique(cut_aggregate, return_counts=True)
    cut_mult_counts = cut_mult_counts / len(cut_aggregate)

    fig, ax = plt.subplots()
    ax.set_yscale("log")
    bars = ax.bar(list(cut_ratio.keys()), list(cut_ratio.values()))
    bar_labels = [(f'{v*100:.2f}%' if v > 0.0023 else "") for v in cut_ratio.values()]
    ax.bar_label(bars, padding=-40, color='white', rotation=90, labels=bar_labels, fontsize=10)
    ax.tick_params(axis='x', labelrotation=90)
    ax.set_ylabel("Fraction of cut waveforms")
    ax.set_ylim(0.001, 0.1)

    fig, ax = plt.subplots()
    ax.set_yscale("log")
    bars = ax.bar(cut_mult_vals, cut_mult_counts)
    bar_labels = [(f'{v*100:.2f}%' if v > 0.0007 else "") for v in cut_mult_counts]
    ax.bar_label(bars, padding=-40, color='white', rotation=90, labels=bar_labels, fontsize=10)
    ax.set_xlabel("Number of SiPMs cut due to noise")
    ax.set_ylabel("Fraction")

    return cut_masks, cut_aggregate

In [ ]:
cut_masks, cut_aggregate = xtalk_stat(dsp_data, usable_sipm_names, 4)

In [ ]:
sipm = "S003"
browser = WaveformBrowser(
    raw_in=[gimme_raw_filename_from_dsp(x) for x in dsp_files],
    lh5_group=f"/{sipm}/raw",
    #entry_list=np.where(cut_masks[sipm]),
    entry_list=np.where(cut_aggregate < 1),
    lines=["waveform_bit_drop"],
    n_drawn=5,
)
browser.draw_next()
[gimme_raw_filename_from_dsp(x) for x in dsp_files]

In [ ]:
def check_sipm(sipm, data, threshold, *, x_lim = None, y_lim = None):
    fig, ax = plt.subplots()
    fig.suptitle(f"min / hwhm, {sipm}")
    ax.set_yscale("log")
    hist, edges = np.histogram(data[f"{sipm}_wf_min_rel"]/data[f"{sipm}_wf_lower_hwhm"], bins=200, range=(-30,10))
    ax.stairs(hist, edges)
    ax.axvline(-threshold, color="black")

    browser = WaveformBrowser(
        raw_in=[gimme_raw_filename_from_dsp(x) for x in dsp_files],
        lh5_group=f"/{sipm}/raw",
        entry_list=np.where(data[f"{sipm}_wf_min_rel"] < -threshold*data[f"{sipm}_wf_lower_hwhm"]),
        #entry_list=np.where(cut_aggregate < 1),
        lines=["waveform_bit_drop"],
        #legend="cut",
        n_drawn=5,
        x_lim=x_lim,
        y_lim=y_lim,
    )
    browser.draw_next()

    browser = WaveformBrowser(
        raw_in=[gimme_raw_filename_from_dsp(x) for x in dsp_files],
        lh5_group=f"/{sipm}/raw",
        entry_list=np.where(
            (data[f"{sipm}_wf_min_rel"] > -threshold*data[f"{sipm}_wf_lower_hwhm"]) & 
            (data[f"{sipm}_wf_min_rel"] < -(threshold-0.5)*data[f"{sipm}_wf_lower_hwhm"])),
        #entry_list=np.where(cut_aggregate < 1),
        lines=["waveform_bit_drop"],
        #legend="just_not_cut",
        n_drawn=5,
        x_lim=x_lim,
        y_lim=y_lim,
    )
    browser.draw_next()

for sipm in usable_sipm_names:
    try:
        check_sipm(sipm, dsp_data, 4, y_lim=(3730,3760), x_lim=(40000,60000))
    except Exception:
        pass